# Notebook 40 - Compression headroom: where does pruning start to cost deployed risk?

**Why this notebook.** On the shallow network, students recovered at 25%, 40% and 55% realised FLOP reduction sit at 0.14 to 0.18 ground-truth risk (HSR), and that risk does not rise across those budgets (notebooks 17 and 19). Recalibrated students of both networks at 40% average 0.15 to 0.19 (notebooks 31 and 37), on average no higher than a dense shallow model trained to the acceptance rule (0.194, notebook 29). At these budgets a new pruning method that promises lower deployed risk has little or nothing to win. Before any method is designed, this notebook finds the budget where compression starts to cost risk, if there is one.

**What it does.** Both CICIoT2023 networks, five selection methods (random, magnitude, Taylor, Fisher, V-C), four realised FLOP budgets (40%, 70%, 80%, 90%). The 40% structures are the frozen structures of notebooks 17b and 20b; the others extend the same removal orders. Inputs are clipped at the 0.1 and 99.9 training percentiles, which removed the BatchNorm collapse in every tested cell (notebooks 35 and 39); the notebook still checks every run for it. Every model, including the dense reference, gets the same recovery and is recalibrated on the common slice before it is audited. The reference is the dense model recovered under the identical seed, so each comparison is paired.

**Three readings, fixed in stage 2 before any result.**
- **H1, headroom.** At some budget of 70%, 80% or 90%, even the best method's recovered ground-truth risk (HSR) is above the dense model's by at least 0.02, with the paired 95% interval above zero.
- **H2, a selection lever.** At a budget where H1 holds, the methods separate: largest paired difference at least 0.02 and at least one pair significant after Holm correction.
- **H3, persistence.** At 90% with standard recovery (two full epochs), the best method is still at least 0.02 above the dense model recovered the same way on average, and above it on every seed.

What each combination means for the method paper is written in the pre-registration (stage 2).

**Stages.** 1 bootstrap, 2 pre-registration, 2b commit of the pre-registration, 3 data, teachers and helpers, 4 structures, 5 minimal-recovery grid (210 runs, resumable), 6 standard-recovery persistence arm (36 runs, resumable), 7 analysis and verdict, 8 figures, 9 commit of the results. GPU; about three hours. If the session stops, rerun from the top: finished runs are skipped.

In [ ]:
# Stage 1 - bootstrap
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, copy, itertools, gc, time, subprocess
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, yaml
import matplotlib.pyplot as plt
from scipy import stats

REPO = Path("/content/drive/MyDrive/SABER_Research/saber-ids")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.saber.bridge_ciciot import load_bridge
from src.saber.deep_model import DeepCNN1D
from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.metrics import full_model_audit, action_weighted_boundary_inversion_rate
from src.saber.surgery import prune_cnn1d_channels, profile_forward_flops, count_parameters
from src.saber.postg5 import calibrate_prefix_count

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
R = REPO / "results/saber"; OUT = R / "40_compression_headroom"; OUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR = REPO / "models/ciciot2023"
print("device:", DEVICE)

In [ ]:
# Stage 2 - pre-registration (frozen: rerunning this cell can never change it after its commit)
PREREG = {
    "notebook": "40_compression_headroom",
    "question": ("at what realised compression budget does a pruned CICIoT2023 detector, recovered and recalibrated, carry more "
                 "ground-truth operational risk than the dense model recovered and recalibrated the same way, and do the selection "
                 "methods separate there"),
    "motivation": ("on the shallow network, students recovered at 25%, 40% and 55% realised FLOP reduction sit at 0.14-0.18 "
                   "ground-truth HSR with stored statistics, and HSR does not rise across those budgets (notebooks 17, 19); "
                   "recalibrated students of both networks at 40% average 0.15-0.19 (notebooks 31, 37), on average no higher than a "
                   "dense shallow model trained to the acceptance rule (0.194, notebook 29); at these budgets a method that lowers "
                   "deployed risk has little or nothing to win; this notebook measures the gap to a dense model recovered the same "
                   "way and looks for the budget where compression starts to cost risk, before any method is designed"),
    "design": {
        "architectures": "shallow CNN (64/128, notebook 17b teacher) and deep CNN (64/128/128/256, notebook 20 teacher), CICIoT2023",
        "methods": "random, magnitude, taylor, fisher, saber_v2 (V-C)",
        "removal_orders": ("ascending score with a per-layer floor (shallow 4 channels, from config/saber.yaml; deep 8), built from the "
                           "committed score tables (shallow: 15_baselines + 16b v_c; deep: 20_depth_probe), the same orders that "
                           "produced the frozen 40% structures; stage 4 asserts that every frozen 40% structure is an exact prefix"),
        "budgets": ("40% (the frozen 17b/20b structures, unchanged) and 70%, 80%, 90% realised FLOP reduction, each prefix calibrated "
                    "with the library routine postg5.calibrate_prefix_count against the surgery profiler, tolerance 1.5 points"),
        "inputs": ("clipped at the 0.1 and 99.9 percentiles of 400,000 training rows (seed 2026), as in notebook 39, for every model "
                   "including the dense reference"),
        "reference": "the dense teacher, recovered and recalibrated exactly like the students under the same seed (paired comparison)",
        "minimal_recovery": ("one pass over a seeded 10% training subset, batch 1,024, Adam 1e-3, inverse-square-root class weights; "
                             "seeds 101, 211, 307, 401, 503; 2 architectures x (1 dense + 5 methods x 4 budgets) x 5 seeds = 210 runs"),
        "standard_recovery": ("two full epochs over all training rows, seeded shuffle, same optimiser and loss; 90% budget only; "
                              "seeds 101, 211, 307; 2 architectures x (1 dense + 5 methods) x 3 seeds = 36 runs"),
        "recalibration": "every recovered model, on the common slice of notebooks 31 to 39 (first 51,200 rows of a seed-2026 permutation), weights frozen",
        "evaluation": "validation evaluation subsample (up to 4,000 rows per class, seed 0), the same rows as notebooks 37 to 39; no test access",
    },
    "endpoint": ("primary: recalibrated ground-truth HSR under the balanced profile; the paired gap of a student is its HSR minus the "
                 "dense reference's HSR under the same seed and regime; benign escalation, attack-miss rate, family and fine macro-F1, "
                 "AWBIR, ECE and per-family recall are recorded and reported descriptively"),
    "H1_headroom": ("for an architecture, H1 holds if at some budget in {70%, 80%, 90%} the BEST method (lowest mean recalibrated HSR "
                    "under minimal recovery) has a mean paired gap of at least 0.02 with the lower end of its paired 95% t interval "
                    "above zero; the smallest such budget is reported as the knee; 0.02 is the within-method seed range of "
                    "recalibrated HSR at 40% (0.014 to 0.023, notebook 37)"),
    "H2_selection_lever": ("for an architecture, H2 holds if at some budget where the H1 condition holds, the largest absolute paired "
                           "difference between two methods' recalibrated HSR is at least 0.02 and at least one of the ten paired "
                           "t-tests is significant after Holm correction (alpha 0.05)"),
    "H3_persistence": ("for an architecture, H3 holds if at 90% under standard recovery the best method (lowest mean recalibrated HSR) "
                       "has a mean paired gap to the dense reference (standard recovery, same seed) of at least 0.02 and a positive "
                       "gap on every one of the three seeds"),
    "readings": {
        "per_architecture": ("no_headroom if H1 fails; otherwise headroom_closes_with_recovery if H3 fails; otherwise "
                             "headroom_without_selection_lever if H2 fails; otherwise method_route_open"),
        "overall": ("the most favourable per-architecture reading, in the order method_route_open, headroom_without_selection_lever, "
                    "headroom_closes_with_recovery, no_headroom"),
        "method_route_open": ("compression costs deployed risk at that budget even after strong recovery and the choice of channels "
                              "changes it: a method paper targets that budget; the first lever to test is layer-width allocation "
                              "(notebook 39)"),
        "headroom_without_selection_lever": ("compression costs risk there but the five methods are equal: a method must act on "
                                             "something other than channel ranking (allocation or recovery); the next test is "
                                             "allocation-matched structures at that budget"),
        "headroom_closes_with_recovery": ("the cost of compression is a recovery-budget effect: a method claim could only be about "
                                          "cheap recovery, which is weaker"),
        "no_headroom": ("compression is free in deployed risk up to 90% on these two networks: the SABER method route is closed on "
                        "this setup, and the manuscript's strongest contribution remains the BatchNorm failure"),
    },
    "seeds_minimal": [101, 211, 307, 401, 503],
    "seeds_standard": [101, 211, 307],
    "budgets": [0.40, 0.70, 0.80, 0.90],
    "gap_threshold": 0.02,
    "no_test_access": True,
}
_text = json.dumps(PREREG, indent=2)
PRE = OUT / "N40_PREREGISTRATION.json"
if PRE.exists() and PRE.read_text() != _text:
    raise RuntimeError("N40_PREREGISTRATION.json already exists with different content. The pre-registration is frozen; restore this cell to the committed version.")
PRE.write_text(_text); print(_text)

ARCHS = ["shallow", "deep"]
METHODS = ["random", "magnitude", "taylor", "fisher", "saber_v2"]
METHOD_SCORE = {"random": "random", "magnitude": "magnitude", "taylor": "taylor", "fisher": "fisher", "saber_v2": "v_c"}
BUDGETS = PREREG["budgets"]; SEEDS = PREREG["seeds_minimal"]; SEEDS_STD = PREREG["seeds_standard"]; GAP = PREREG["gap_threshold"]
SUBSET_FRACTION = 0.10; CAL_SEED = 2026; TOL = 0.015; STD_EPOCHS = 2; COLLAPSE_B2A = 0.10
MIN_W = {"shallow": int(yaml.safe_load(open(REPO / "config/saber.yaml"))["groups"]["minimum_remaining_per_layer"]), "deep": 8}
print("per-layer floor:", MIN_W)

In [ ]:
# Stage 2b - commit the pre-registration before any result exists (skipped automatically once results exist)
def _prereg_committed():
    rel = "results/saber/40_compression_headroom/N40_PREREGISTRATION.json"
    h = subprocess.run(["git", "log", "-1", "--format=%h", "--", rel], capture_output=True, text=True, cwd=REPO).stdout.strip()
    dirty = subprocess.run(["git", "status", "--porcelain", "--", rel], capture_output=True, text=True, cwd=REPO).stdout.strip()
    return bool(h) and not dirty


_results_exist = any(p.name != "N40_PREREGISTRATION.json" for p in OUT.iterdir())
if _results_exist:
    assert _prereg_committed(), ("results exist but the pre-registration is not committed, so it can no longer be committed before any "
                                 "result; stop here and report it")
    print("results already exist in", OUT.name, "and the pre-registration is committed; nothing to do here")
else:
    os.chdir(REPO)
    r = subprocess.run(["python", "tools/commit_cell.py",
                        "NB40 pre-registration, committed before any result: compression headroom on CICIoT2023. Both networks, five "
                        "methods, realised FLOP budgets 40/70/80/90% (40% = frozen 17b/20b structures, others extend the same removal "
                        "orders), clipped inputs, dense reference recovered under the same seed, every model recalibrated; minimal "
                        "recovery 5 seeds (210 runs) and a standard-recovery persistence arm at 90% with 3 seeds (36 runs). Readings H1 "
                        "headroom, H2 selection lever, H3 persistence, threshold 0.02 recalibrated HSR."],
                       capture_output=True, text=True)
    print(r.stdout); print(r.stderr)
    assert _prereg_committed(), "the pre-registration commit did not go through; fix it before running stage 3"
    print("pre-registration committed:", subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True, cwd=REPO).stdout.strip()[:120])

In [ ]:
# Stage 3 - data, teachers and helpers
TRAIN_LOADER, VAL_LOADER, _T, SHALLOW_TEACHER, CLASS_NAMES = load_bridge()
taxonomy = ciciot2023_taxonomy(CLASS_NAMES); robust_graph = pd.read_csv(R / "14_risk_graph/asvg_edges_robust.csv")
N_CLASSES = len(CLASS_NAMES)
TEACHERS = {"shallow": SHALLOW_TEACHER.to(DEVICE).eval()}
_dt = DeepCNN1D(N_CLASSES); _dt.load_state_dict(torch.load(MODEL_DIR / "deepcnn1d_g5_seed0.pt", map_location="cpu", weights_only=False)["state_dict"])
TEACHERS["deep"] = _dt.to(DEVICE).eval()

Xv, Yv = VAL_LOADER.dataset.tensors; VAL_Y_ALL = Yv.numpy(); Xt, Yt = TRAIN_LOADER.dataset.tensors; N_TRAIN = len(Xt)
_rng = np.random.default_rng(0)
_idx = np.concatenate([_rng.permutation(np.where(VAL_Y_ALL == c)[0])[:4000] for c in range(N_CLASSES) if (VAL_Y_ALL == c).sum() > 0])
_idx = np.random.default_rng(12345).permutation(_idx)
EX_X = Xv[_idx].to(DEVICE); EX_Y = VAL_Y_ALL[_idx]; EXAMPLE_INPUT = Xv[:8].float().to(DEVICE)
_counts = np.bincount(Yt.numpy(), minlength=N_CLASSES); _w = np.zeros_like(_counts, dtype=np.float64)
_w[_counts > 0] = 1.0 / np.sqrt(_counts[_counts > 0]); _w[_counts > 0] /= _w[_counts > 0].mean()
CLASS_W = torch.tensor(_w, dtype=torch.float32, device=DEVICE)
_cal_idx = torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(CAL_SEED))[:50 * 1024]
CAL_LOADER = torch.utils.data.DataLoader(torch.utils.data.Subset(TRAIN_LOADER.dataset, _cal_idx.tolist()), batch_size=1024, shuffle=False)
_sub = Xt[torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(2026))[:400_000]].numpy()
LO = torch.tensor(np.percentile(_sub, 0.1, axis=0), dtype=torch.float32, device=DEVICE); HI = torch.tensor(np.percentile(_sub, 99.9, axis=0), dtype=torch.float32, device=DEVICE)
del _sub
FAM = np.array(taxonomy.family_by_class); FAMILIES = sorted(set(FAM.tolist())); FAM_Y = FAM[EX_Y]


class Clipped(nn.Module):
    def __init__(self, inner, lo, hi):
        super().__init__(); self.inner = inner; self.register_buffer("lo", lo.clone()); self.register_buffer("hi", hi.clone())
    def forward(self, x):
        return self.inner(torch.maximum(torch.minimum(x, self.hi), self.lo))


def forward_logits(model, X=None):
    X = EX_X if X is None else X; model.eval()
    with torch.no_grad():
        return torch.cat([model(X[i:i + 8192]).cpu() for i in range(0, len(X), 8192)]).numpy()


def forward_logits_batch_stats(model):
    # same weights, BatchNorm normalising by each batch's own statistics; buffers saved and restored (notebook 39)
    saved = {n: (m.running_mean.clone(), m.running_var.clone(), m.momentum, m.num_batches_tracked.clone()) for n, m in model.named_modules() if isinstance(m, nn.BatchNorm1d) and m.running_mean is not None}
    model.train()
    for n, m in model.named_modules():
        if n in saved: m.momentum = 0.0
    with torch.no_grad():
        out = torch.cat([model(EX_X[i:i + 8192]).cpu() for i in range(0, len(EX_X), 8192)]).numpy()
    for n, m in model.named_modules():
        if n in saved:
            rm, rv, mom, nbt = saved[n]; m.running_mean.copy_(rm); m.running_var.copy_(rv); m.momentum = mom; m.num_batches_tracked.copy_(nbt)
    model.eval(); return out


T_LOGITS = {a: forward_logits(m) for a, m in TEACHERS.items()}  # unclipped teachers, the AWBIR reference as in notebooks 37 to 39


def audit(model, arch, lg=None):
    lg = forward_logits(model) if lg is None else lg; a = full_model_audit(lg, EX_Y, taxonomy, DEFAULT_COST_PROFILES)
    aw, _ = action_weighted_boundary_inversion_rate(T_LOGITS[arch], lg, EX_Y, robust_graph)
    pf = FAM[lg.argmax(axis=1)]
    out = {"b2a": float(a["benign_to_attack_rate"]), "a2b": float(a["attack_to_benign_rate"]), "family_f1": float(a["family_macro_f1"]),
           "fine_f1": float(a["fine_macro_f1"]), "hsr_balanced": float(a["hsr_balanced_soc"]), "hsr_miss": float(a["hsr_miss_sensitive"]),
           "hsr_fatigue": float(a["hsr_alert_fatigue"]), "awbir": float(aw), "ece": float(a["ece15"])}
    out.update({f"recall_{f}": float((pf[FAM_Y == f] == f).mean()) for f in FAMILIES})
    return out


def recalibrate_bn(model, n_batches=50):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d): m.reset_running_stats(); m.momentum = None
    model.train()
    with torch.no_grad():
        for i, (xb, _) in enumerate(CAL_LOADER):
            if i >= n_batches: break
            model(xb.to(DEVICE))
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d): m.momentum = 0.1
    model.eval(); return model


def make_minimal_loader(seed):
    sub = torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(seed))[: int(N_TRAIN * SUBSET_FRACTION)]
    return torch.utils.data.DataLoader(torch.utils.data.Subset(TRAIN_LOADER.dataset, sub.tolist()), batch_size=1024, shuffle=True, generator=torch.Generator().manual_seed(seed))


def make_full_loader(seed):
    return torch.utils.data.DataLoader(TRAIN_LOADER.dataset, batch_size=1024, shuffle=True, generator=torch.Generator().manual_seed(seed))


def recover(model, loader, epochs):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss(weight=CLASS_W)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad(); lossf(model(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
    return model.eval()


# context only: the two teachers as they are, unclipped and clipped, before any recovery
ctx = []
for arch, m in TEACHERS.items():
    ctx.append({"architecture": arch, "model": "teacher, unclipped, no recovery", **audit(m, arch)})
    ctx.append({"architecture": arch, "model": "teacher, clipped, no recovery", **audit(Clipped(copy.deepcopy(m), LO, HI).to(DEVICE), arch)})
pd.DataFrame(ctx).to_csv(OUT / "N40_teachers.csv", index=False)
print(pd.DataFrame(ctx)[["architecture", "model", "hsr_balanced", "b2a", "a2b", "family_f1", "fine_f1"]].round(4).to_string(index=False))
print("CICIoT2023 ready | train", N_TRAIN, "| eval rows", len(_idx), "| clip bounds from", min(N_TRAIN, 400_000), "training rows")

In [ ]:
# Stage 4 - structures: removal orders from the committed score tables; 40% = frozen structures; 70/80/90% calibrated
SCORES = {"shallow": pd.read_csv(R / "15_baselines/group_baseline_scores.csv").merge(pd.read_csv(R / "16b_score_v2/v2_scores.csv")[["group_id", "v_c"]], on="group_id", validate="1:1"),
          "deep": pd.read_csv(R / "20_depth_probe/deep_group_scores.csv")}
FROZEN = {"shallow": lambda m: R / f"17b_calibrated_checkpoint_freeze/{m}_r40cal_removed_groups.csv",
          "deep": lambda m: R / f"20b_depth_checkpoint_freeze/{m}_minimal_r40_removed_groups.csv"}
M0 = {a: profile_forward_flops(TEACHERS[a], EXAMPLE_INPUT)["flops_per_item"] for a in ARCHS}
P0 = {a: count_parameters(TEACHERS[a]) for a in ARCHS}


def valid_order(arch, column):
    table = SCORES[arch].sort_values(column, ascending=True)
    remaining = table.groupby("module_path")["group_id"].count().to_dict(); order = []
    for row in table.itertuples():
        layer = str(row.module_path)
        if remaining[layer] - 1 < MIN_W[arch]: continue
        remaining[layer] -= 1; order.append((layer, int(row.channel_index)))
    return order


def prune_map_of(pairs):
    pm = {}
    for layer, ch in pairs: pm.setdefault(str(layer), []).append(int(ch))
    return {l: sorted(c) for l, c in pm.items()}


def build_inner(arch, pm):
    st, _ = prune_cnn1d_channels(TEACHERS[arch], pm, EXAMPLE_INPUT, minimum_remaining_per_layer=MIN_W[arch]); return st.to(DEVICE)


def realised(arch, st):
    return 1.0 - profile_forward_flops(st, EXAMPLE_INPUT)["flops_per_item"] / M0[arch]


def widths(st):
    return "/".join(str(m.out_channels) for _, m in st.named_modules() if isinstance(m, nn.Conv1d))


def removed_file(arch, method, budget):
    return OUT / f"N40_{arch}_{method}_b{int(round(budget * 100))}_removed_groups.csv"


struct_rows, traces = [], []
for arch in ARCHS:
    for method in METHODS:
        order = valid_order(arch, METHOD_SCORE[method])
        fr = pd.read_csv(FROZEN[arch](method)); frozen_pairs = list(zip(fr.module_path.astype(str), fr.channel_index.astype(int)))
        assert order[:len(frozen_pairs)] == frozen_pairs, f"{arch}/{method}: the frozen 40% structure is not a prefix of the rebuilt order"
        for budget in BUDGETS:
            if np.isclose(budget, 0.40):
                k, source = len(frozen_pairs), "frozen"
            else:
                cal = calibrate_prefix_count(order, target_reduction=budget, tolerance=TOL,
                                             build_model=lambda k, a=arch, o=order: build_inner(a, prune_map_of(o[:k])),
                                             realised_reduction=lambda st, a=arch: realised(a, st))
                k, source = cal.prefix_length, "calibrated"
                traces += [{"architecture": arch, "method": method, "budget": budget, **t} for t in cal.trace]
            st = build_inner(arch, prune_map_of(order[:k]))
            pd.DataFrame([{"module_path": l, "channel_index": c} for l, c in order[:k]]).to_csv(removed_file(arch, method, budget), index=False)
            raw = audit(Clipped(st, LO, HI).to(DEVICE), arch); rr = realised(arch, st)
            struct_rows.append({"architecture": arch, "method": method, "budget": budget, "source": source, "k": k, "realised_reduction": rr,
                                "within_tolerance": bool(abs(rr - budget) <= TOL), "widths": widths(st), "parameters": int(count_parameters(st)),
                                "parameter_ratio": count_parameters(st) / P0[arch], **{f"raw_{kk}": v for kk, v in raw.items()}})
            print(f"{arch:7s} {method:9s} {budget:.2f} [{source:10s}] k={k:3d} realised={rr:.4f} widths={widths(st):16s} params={struct_rows[-1]['parameters']:6d} raw HSR={raw['hsr_balanced']:.4f} raw b2a={raw['b2a']:.3f}")
STRUCT = pd.DataFrame(struct_rows); STRUCT.to_csv(OUT / "N40_structures.csv", index=False); pd.DataFrame(traces).to_csv(OUT / "N40_prefix_search.csv", index=False)
assert STRUCT.within_tolerance.all(), "a structure is outside the 1.5-point tolerance:\n" + STRUCT[~STRUCT.within_tolerance].to_string()
print("all", len(STRUCT), "structures within tolerance")

In [ ]:
# Stage 5 - minimal recovery grid: dense reference + 5 methods x 4 budgets, 5 seeds, both networks (resumable)
RUNS = OUT / "N40_runs.csv"
rows = pd.read_csv(RUNS).to_dict("records") if RUNS.exists() else []
done = {(r["regime"], r["architecture"], round(float(r["budget"]), 2), r["method"], int(r["seed"])) for r in rows}


def student_for(arch, method, budget):
    if method == "dense":
        return copy.deepcopy(TEACHERS[arch]).to(DEVICE)
    rm = pd.read_csv(removed_file(arch, method, budget))
    return build_inner(arch, prune_map_of(zip(rm.module_path.astype(str), rm.channel_index.astype(int))))


def run_one(regime, arch, budget, method, seed):
    torch.manual_seed(seed); np.random.seed(seed); t0 = time.time()
    model = Clipped(student_for(arch, method, budget), LO, HI).to(DEVICE)
    loader = make_minimal_loader(seed) if regime == "minimal" else make_full_loader(seed)
    recover(model, loader, 1 if regime == "minimal" else STD_EPOCHS)
    tr = audit(model, arch); bs = audit(model, arch, lg=forward_logits_batch_stats(model)); recalibrate_bn(model); rc = audit(model, arch)
    return {"regime": regime, "architecture": arch, "budget": budget, "method": method, "seed": seed, "parameters": int(count_parameters(model.inner)),
            "batch_b2a": bs["b2a"], "collapse_as_trained": bool(tr["b2a"] > COLLAPSE_B2A and bs["b2a"] <= COLLAPSE_B2A), "seconds": round(time.time() - t0, 1),
            **{f"trained_{k}": v for k, v in tr.items()}, **{f"recal_{k}": v for k, v in rc.items()}}


def run_jobs(jobs):
    for job in jobs:
        key = (job[0], job[1], round(float(job[2]), 2), job[3], int(job[4]))
        if key in done: continue
        row = run_one(*job); rows.append(row); done.add(key); pd.DataFrame(rows).to_csv(RUNS, index=False)
        print(f"{job[0]:8s} {job[1]:7s} b={job[2]:.2f} {job[3]:9s} s{job[4]:4d}: recal HSR {row['recal_hsr_balanced']:.4f} b2a {row['recal_b2a']:.4f} "
              f"a2b {row['recal_a2b']:.4f} famF1 {row['recal_family_f1']:.3f} | {row['seconds']:.0f}s")


JOBS_MIN = ([("minimal", a, 0.0, "dense", s) for a in ARCHS for s in SEEDS]
            + [("minimal", a, b, m, s) for a in ARCHS for b in BUDGETS for m in METHODS for s in SEEDS])
run_jobs(JOBS_MIN)
print("minimal runs:", sum(r["regime"] == "minimal" for r in rows), "of", len(JOBS_MIN))

In [ ]:
# Stage 6 - persistence arm: standard recovery (two full epochs) at 90%, dense reference + 5 methods, 3 seeds (resumable)
JOBS_STD = ([("standard", a, 0.0, "dense", s) for a in ARCHS for s in SEEDS_STD]
            + [("standard", a, 0.90, m, s) for a in ARCHS for m in METHODS for s in SEEDS_STD])
run_jobs(JOBS_STD)
print("standard runs:", sum(r["regime"] == "standard" for r in rows), "of", len(JOBS_STD))

In [ ]:
# Stage 7 - analysis and verdict (rules exactly as pre-registered in stage 2)
runs = pd.read_csv(RUNS); STRUCT = pd.read_csv(OUT / "N40_structures.csv")
assert len(runs) == len(JOBS_MIN) + len(JOBS_STD), f"expected {len(JOBS_MIN) + len(JOBS_STD)} runs, found {len(runs)}: finish stages 5 and 6 first"
COL = "recal_hsr_balanced"


def series(regime, arch, budget, method, col=COL):
    q = runs[(runs.regime == regime) & (runs.architecture == arch) & (runs.method == method)]
    if method != "dense": q = q[np.isclose(q.budget, budget)]
    return q.set_index("seed")[col].sort_index()


def paired_gap(regime, arch, budget, method, col=COL):
    s = series(regime, arch, budget, method, col); d = series(regime, arch, 0.0, "dense", col); common = s.index.intersection(d.index)
    g = (s.loc[common] - d.loc[common]).to_numpy(); n = len(g); hw = stats.t.ppf(0.975, n - 1) * g.std(ddof=1) / np.sqrt(n)
    return {"n": int(n), "gap": float(g.mean()), "gap_lo": float(g.mean() - hw), "gap_hi": float(g.mean() + hw), "min_seed_gap": float(g.min())}


def holm(ps):
    order = np.argsort(ps); out = np.empty(len(ps)); running = 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, ps[i] * (len(ps) - rank))); out[i] = running
    return out


def contrasts(regime, arch, budget):
    piv = pd.DataFrame({m: series(regime, arch, budget, m) for m in METHODS}).dropna()
    rows_ = []
    for a, b in itertools.combinations(METHODS, 2):
        d = (piv[a] - piv[b]).to_numpy(); rows_.append({"regime": regime, "architecture": arch, "budget": budget, "contrast": f"{a}-{b}", "mean": float(d.mean()), "p": float(stats.ttest_rel(piv[a], piv[b]).pvalue)})
    ph = holm(np.array([r["p"] for r in rows_]))
    for r, p in zip(rows_, ph): r["p_holm"] = float(p); r["significant_holm"] = bool(p < 0.05)
    return rows_


METRICS = ["recal_hsr_balanced", "recal_b2a", "recal_a2b", "recal_family_f1", "recal_fine_f1", "recal_awbir", "recal_ece", "trained_hsr_balanced", "trained_b2a"]
summary = []
for regime, budgets in [("minimal", BUDGETS), ("standard", [0.90])]:
    for arch in ARCHS:
        d = runs[(runs.regime == regime) & (runs.architecture == arch) & (runs.method == "dense")]
        summary.append({"regime": regime, "architecture": arch, "budget": 0.0, "method": "dense", "n": len(d), "hsr_sd": float(d[COL].std(ddof=1)), **{m: float(d[m].mean()) for m in METRICS}})
        for b in budgets:
            for m in METHODS:
                q = runs[(runs.regime == regime) & (runs.architecture == arch) & (runs.method == m) & np.isclose(runs.budget, b)]
                summary.append({"regime": regime, "architecture": arch, "budget": b, "method": m, "n": len(q), "hsr_sd": float(q[COL].std(ddof=1)),
                                **{mm: float(q[mm].mean()) for mm in METRICS}, **paired_gap(regime, arch, b, m)})
SUM = pd.DataFrame(summary); SUM.to_csv(OUT / "N40_summary.csv", index=False)

H, CON = {}, []
for arch in ARCHS:
    h1_budgets, h1_detail = [], {}
    for b in [x for x in BUDGETS if x > 0.40 + 1e-9]:
        means = {m: float(series("minimal", arch, b, m).mean()) for m in METHODS}; best = min(means, key=means.get); g = paired_gap("minimal", arch, b, best)
        cond = bool(g["gap"] >= GAP and g["gap_lo"] > 0); h1_detail[f"{b:.2f}"] = {"best_method": best, **g, "condition": cond}
        if cond: h1_budgets.append(b)
    h2_detail = {}
    for b in h1_budgets:
        cs = contrasts("minimal", arch, b); CON += cs
        mx = max(abs(c["mean"]) for c in cs); nsig = sum(c["significant_holm"] for c in cs)
        h2_detail[f"{b:.2f}"] = {"max_abs_paired_difference": float(mx), "n_holm_significant": int(nsig), "condition": bool(mx >= GAP and nsig >= 1)}
    for b in BUDGETS:
        if b not in h1_budgets: CON += contrasts("minimal", arch, b)
    means90 = {m: float(series("standard", arch, 0.90, m).mean()) for m in METHODS}; best90 = min(means90, key=means90.get); g90 = paired_gap("standard", arch, 0.90, best90)
    H1 = bool(len(h1_budgets) > 0); H2 = bool(any(v["condition"] for v in h2_detail.values())); H3 = bool(g90["gap"] >= GAP and g90["min_seed_gap"] > 0)
    reading = "no_headroom" if not H1 else ("headroom_closes_with_recovery" if not H3 else ("headroom_without_selection_lever" if not H2 else "method_route_open"))
    anchor = {m: paired_gap("minimal", arch, 0.40, m)["gap"] for m in METHODS}
    H[arch] = {"H1_headroom": H1, "knee_budget": (min(h1_budgets) if H1 else None), "H1_by_budget": h1_detail,
               "H2_selection_lever": H2, "H2_by_budget": h2_detail,
               "H3_persistence": H3, "H3_standard_90": {"best_method": best90, **g90, "means": means90},
               "anchor_gap_at_40": anchor, "reading": reading}
pd.DataFrame(CON).to_csv(OUT / "N40_contrasts.csv", index=False)
RANK = ["method_route_open", "headroom_without_selection_lever", "headroom_closes_with_recovery", "no_headroom"]
overall = min((H[a]["reading"] for a in ARCHS), key=RANK.index)
verdict = {"notebook": "40_compression_headroom", "reading": overall, "meaning": PREREG["readings"][overall], "by_architecture": H,
           "collapses_as_trained": int(runs.collapse_as_trained.sum()), "prereg": PREREG}
(OUT / "N40_verdict.json").write_text(json.dumps(verdict, indent=2, default=float))

print("OVERALL READING:", overall, "-", PREREG["readings"][overall])
for arch in ARCHS:
    h = H[arch]
    print(f"\n{arch}: H1 {h['H1_headroom']} (knee {h['knee_budget']}) | H2 {h['H2_selection_lever']} | H3 {h['H3_persistence']} -> {h['reading']}")
    for b, v in h["H1_by_budget"].items():
        print(f"  {b}: best {v['best_method']:9s} paired gap {v['gap']:+.4f} [{v['gap_lo']:+.4f}, {v['gap_hi']:+.4f}]")
    v = h["H3_standard_90"]; print(f"  standard 0.90: best {v['best_method']:9s} paired gap {v['gap']:+.4f}, smallest seed gap {v['min_seed_gap']:+.4f}")
    print("  gaps at 40% (anchor):", {m: round(x, 4) for m, x in h["anchor_gap_at_40"].items()})
print("\nas-trained collapses (clipped inputs, expected 0):", verdict["collapses_as_trained"])
show = SUM[["regime", "architecture", "budget", "method", "recal_hsr_balanced", "hsr_sd", "gap", "gap_lo", "gap_hi", "recal_b2a", "recal_a2b", "recal_family_f1"]]
print(show.round(4).to_string(index=False))

In [ ]:
# Stage 8 - figures
runs = pd.read_csv(OUT / "N40_runs.csv")
XSTD = 0.98  # x position of the standard-recovery points (all at the 90% budget)
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for j, arch in enumerate(ARCHS):
    for i, (col, label) in enumerate([("recal_hsr_balanced", "ground-truth HSR, recalibrated"), ("recal_family_f1", "family macro-F1, recalibrated")]):
        ax = axes[i, j]
        dense = runs[(runs.regime == "minimal") & (runs.architecture == arch) & (runs.method == "dense")][col]
        ax.axhspan(dense.mean() - dense.std(ddof=1), dense.mean() + dense.std(ddof=1), color="0.85", label="dense, minimal recovery (mean +/- sd)")
        ax.axhline(dense.mean(), color="0.4", lw=1)
        for k, m in enumerate(METHODS):
            q = runs[(runs.regime == "minimal") & (runs.architecture == arch) & (runs.method == m)].groupby("budget")[col]
            x = np.array(q.mean().index) + (k - 2) * 0.006
            eb = ax.errorbar(x, q.mean().values, yerr=q.std(ddof=1).values, marker="o", ms=4, capsize=2, label=f"{m}, minimal recovery")
            s = runs[(runs.regime == "standard") & (runs.architecture == arch) & (runs.method == m)][col]
            ax.scatter([XSTD + (k - 2) * 0.006], [s.mean()], marker="D", facecolors="none", edgecolors=eb[0].get_color(), s=30)
        ds = runs[(runs.regime == "standard") & (runs.architecture == arch) & (runs.method == "dense")][col]
        ax.scatter([XSTD], [ds.mean()], marker="*", color="0.2", s=90, label="90%, standard recovery: dense (star), methods (diamonds)")
        ax.axvline((0.90 + XSTD) / 2 + 0.005, color="0.7", lw=0.8, ls=":")
        ax.set_title(f"{arch} network", fontsize=10); ax.set_ylabel(label)
        if i == 1: ax.set_xlabel("realised FLOP reduction")
        ax.set_xlim(0.36, XSTD + 0.03)
        ax.set_xticks(BUDGETS + [XSTD]); ax.set_xticklabels([f"{int(b * 100)}%" for b in BUDGETS] + ["90%\nstandard"])
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, fontsize=8, frameon=False)
fig.tight_layout(rect=[0, 0.08, 1, 1]); fig.savefig(OUT / "N40_headroom.png", dpi=200); plt.show(); print("written ->", OUT)

In [ ]:
# Stage 9 - commit the results (standard commit cell; the message is filled in from the verdict)
v = json.load(open(OUT / "N40_verdict.json"))
pre = subprocess.run(["git", "log", "-1", "--format=%h", "--", "results/saber/40_compression_headroom/N40_PREREGISTRATION.json"], capture_output=True, text=True, cwd=REPO).stdout.strip() or "see log"
parts = []
for arch in ARCHS:
    h = v["by_architecture"][arch]; b90 = h["H1_by_budget"]["0.90"]; s90 = h["H3_standard_90"]
    parts.append(f"{arch}: reading {h['reading']}; H1 {'HELD' if h['H1_headroom'] else 'FAILED'}"
                 + (f" (knee {h['knee_budget']:.2f})" if h["H1_headroom"] else "")
                 + f", best method at 90% minimal {b90['best_method']} paired HSR gap {b90['gap']:+.3f} [{b90['gap_lo']:+.3f}, {b90['gap_hi']:+.3f}]"
                 + f"; H2 {'HELD' if h['H2_selection_lever'] else 'FAILED'}; H3 {'HELD' if h['H3_persistence'] else 'FAILED'}"
                 + f" (standard 90% best {s90['best_method']} gap {s90['gap']:+.3f}, smallest seed gap {s90['min_seed_gap']:+.3f}).")
msg = (f"NB40 results (pre-registration {pre}): compression headroom on CICIoT2023, overall reading {v['reading']}. " + " ".join(parts)
       + f" As-trained collapses with clipped inputs: {v['collapses_as_trained']}.")
print(msg)
os.chdir(REPO)
r = subprocess.run(["python", "tools/commit_cell.py", msg], capture_output=True, text=True)
print(r.stdout); print(r.stderr)